# Enveda CASMI 2026 — submission v17（FPNet 指纹预测 + fz 排序 + 扩展候选池）

v7=0.231（library + COCONUT mass-only）。v17 在 v7 基础上扩展候选池：
- prvsiyan 的 FPNet（transformer）：谱图 → 6930-bit 分子指纹 logits
- 候选池：COCONUT 436K + PubChemLite 386K + ChEBI/LipidMaps 32K + NPAtlas 2K + HMDB 124K = 980K
- 排序：fz = candidate_fp_bits @ z（logits 点积），±10ppm 质量窗口
- 训练库结构：RDKit 现场指纹化后并入候选池（cp313 dataset；挂载失败则跳过）

In [ ]:
import os, sys, time, glob as _glob, pickle, math
import numpy as np, pandas as pd, pyarrow.parquet as pq
t1 = time.time()

def _find_data_dir():
    for c in ["/kaggle/input/competitions/enveda-CASMI26-molecule-id-mass-spectra",
              "/kaggle/input/enveda-CASMI26-molecule-id-mass-spectra"]:
        import os as _os
        if _os.path.isdir(c):
            return c
    raise RuntimeError("competition data dir not found")
DATA = _find_data_dir()
def _find_fp_dir():
    for c in ["/kaggle/input/datasets/dmitriigluzdov/casmi26-open-model-and-candidate-resources",
              "/kaggle/input/casmi26-open-model-and-candidate-resources"]:
        if os.path.isfile(os.path.join(c, "fp_single_s2.pt")):
            return c
    raise RuntimeError("fp dataset dir not found; checked standard kaggle mount points")
FPD = _find_fp_dir()
print("fp dir:", FPD, flush=True)

def _find_expanded_dir():
    """Find the expanded pool dataset (v17: ChEBI/LipidMaps + NPAtlas + HMDB)."""
    for c in ["/kaggle/input/enveda-expanded-pool-v17",
              "/kaggle/input/datasets/naijilu/enveda-expanded-pool-v17",
              "/kaggle/input/enveda-expanded-pool"]:
        if os.path.isfile(os.path.join(c, "hmdb_new_fp.npy")):
            return c
    print("WARNING: expanded pool dataset not found; using only base pools", flush=True)
    return None
EXPD = _find_expanded_dir()
if EXPD:
    print("expanded pool dir:", EXPD, flush=True)

In [ ]:
# ---------- FPNet（架构逐字取自 dmitriigluzdov 公开 notebook） ----------
import torch, torch.nn as nn, torch.nn.functional as F

FP_MAX_PEAKS = 128
ADDUCT_LIST = ["[M+H]+", "[M+NH4]+", "[M+Na]+", "[M+K]+", "[M-H2O+H]+", "[M-2H2O+H]+", "[M]+",
               "[M-H]-", "[M-H2O-H]-", "[M+CH2O2-H]-", "[M+C2H4O2-H]-", "[M+Cl]-", "[M]-",
               "[M+2H]2+", "[M-2H]-", "[2M+H]+", "[2M+Na]+", "[2M+NH4]+", "[2M-H]-", "[2M+K]+",
               "[2M+CH2O2-H]-", "[2M+C2H4O2-H]-", "[2M+Na-2H]-", "[M+Na-2H]-", "[M-H2O]+", "<unk>"]
ADDUCT_IX = {a: i for i, a in enumerate(ADDUCT_LIST)}

def instr_family(s):
    if s is None: return 4
    t = str(s).lower()
    if 'timstof' in t: return 0
    if 'orbitrap' in t or 'qft' in t or 'ftms' in t or 'hybrid ft' in t or 'itft' in t or 'exactive' in t: return 1
    if 'tof' in t: return 2
    if 'trap' in t or 'qq' in t: return 3
    return 4

def prep_peaks(mz, inten, prec_mz, max_peaks=FP_MAX_PEAKS, floor=1e-3, win=50.0, per_win=8):
    mz = np.asarray(mz, np.float64); it = np.asarray(inten, np.float64)
    if len(mz) == 0: return np.zeros(0, np.float32), np.zeros(0, np.float32)
    keep = (mz <= prec_mz + 1.5)
    mz, it = mz[keep], it[keep]
    if len(mz) == 0: return np.zeros(0, np.float32), np.zeros(0, np.float32)
    mx = it.max()
    if mx <= 0: return np.zeros(0, np.float32), np.zeros(0, np.float32)
    keep = it >= floor * mx
    mz, it = mz[keep], it[keep]
    if len(mz) > max_peaks:
        order = np.argsort(-it)
        bucket = (mz // win).astype(np.int64)
        cnt = {}; sel = []
        for i in order:
            b = bucket[i]; c = cnt.get(b, 0)
            if c < per_win: cnt[b] = c + 1; sel.append(i)
        sel = np.array(sel)
        if len(sel) > max_peaks:
            sel = sel[np.argsort(-it[sel])[:max_peaks]]
        elif len(sel) < max_peaks:
            rest = np.array([i for i in order if i not in set(sel.tolist())])
            need = max_peaks - len(sel)
            if len(rest): sel = np.concatenate([sel, rest[:need]])
        mz, it = mz[sel], it[sel]
    o = np.argsort(mz)
    mz, it = mz[o], it[o]
    v = np.sqrt(it / it.max())
    return mz.astype(np.float32), v.astype(np.float32)

class SinEmb(nn.Module):
    def __init__(self, dim, lo=-2.0, hi=3.2, power=1.0):
        super().__init__()
        n = dim // 2
        wav = torch.pow(10.0, (hi - lo) * torch.pow(torch.linspace(0, 1, n), power) + lo)
        self.register_buffer('inv', (2 * math.pi) / wav)
    def forward(self, x):
        a = x.unsqueeze(-1) * self.inv
        return torch.cat([torch.sin(a), torch.cos(a)], -1)

class Block(nn.Module):
    def __init__(self, d, h, drop):
        super().__init__(); self.h = h
        self.n1 = nn.LayerNorm(d); self.qkv = nn.Linear(d, 3 * d); self.o = nn.Linear(d, d)
        self.n2 = nn.LayerNorm(d)
        self.ff = nn.Sequential(nn.Linear(d, 4 * d), nn.GELU(), nn.Dropout(drop), nn.Linear(4 * d, d))
        self.drop = nn.Dropout(drop)
    def forward(self, x, pad):
        B, N, D = x.shape; y = self.n1(x)
        q, k, v = self.qkv(y).view(B, N, 3, self.h, D // self.h).permute(2, 0, 3, 1, 4)
        m = (~pad)[:, None, None, :]
        a = F.scaled_dot_product_attention(q, k, v, attn_mask=m)
        x = x + self.drop(self.o(a.transpose(1, 2).reshape(B, N, D)))
        return x + self.drop(self.ff(self.n2(x)))

class FPNet(nn.Module):
    def __init__(self, nbits, d=512, layers=6, heads=8, drop=0.1):
        super().__init__()
        self.d = d
        self.mz_emb = SinEmb(d)
        self.nl_emb = SinEmb(d)
        self.pk = nn.Linear(2 * d + 1, d)
        self.prec_emb = SinEmb(d)
        self.ad = nn.Embedding(len(ADDUCT_LIST), d)
        self.ins = nn.Embedding(5, d)
        self.gl = nn.Linear(d + 3, d)
        self.blocks = nn.ModuleList([Block(d, heads, drop) for _ in range(layers)])
        self.norm = nn.LayerNorm(d)
        self.head = nn.Sequential(nn.Linear(2 * d, 2048), nn.GELU(), nn.Dropout(drop), nn.Linear(2048, nbits))
    def forward(self, mz, it, pad, prec, ad, ins, ce, mode):
        B, N = mz.shape
        nl = (prec[:, None] - mz).clamp(min=0)
        p = self.pk(torch.cat([self.mz_emb(mz), self.nl_emb(nl), it.unsqueeze(-1)], -1))
        g = self.gl(torch.cat([self.prec_emb(prec),
                               (ce / 100.0).unsqueeze(-1), mode.unsqueeze(-1),
                               torch.log1p(prec).unsqueeze(-1) / 10.0], -1)) + self.ad(ad) + self.ins(ins)
        x = torch.cat([g.unsqueeze(1), p], 1)
        pad = torch.cat([torch.zeros(B, 1, dtype=torch.bool, device=pad.device), pad], 1)
        for b in self.blocks: x = b(x, pad)
        x = self.norm(x)
        cls = x[:, 0]
        msk = (~pad[:, 1:]).float().unsqueeze(-1)
        mean = (x[:, 1:] * msk).sum(1) / msk.sum(1).clamp(min=1)
        return self.head(torch.cat([cls, mean], -1))

def load_fp_models(paths, dev):
    single, merged = [], []
    nbits = None
    for pth in paths:
        ck = torch.load(pth, map_location='cpu', weights_only=False)
        net = FPNet(ck['nbits'], d=ck['d'], layers=ck['layers']).to(dev).eval()
        net.load_state_dict(ck['model'])
        (merged if 'merged' in str(pth).split('/')[-1] else single).append(net)
        nbits = ck['nbits']
    return single, merged, nbits

def merge_peaks(specs):
    specs = [(a, b) for a, b in specs if len(a)]
    if not specs: return np.zeros(0), np.zeros(0)
    mz = np.concatenate([np.asarray(a, float) for a, _ in specs])
    it = np.concatenate([np.asarray(b, float) / max(float(np.asarray(b, float).max()), 1e-9) for _, b in specs])
    o = np.argsort(mz); mz, it = mz[o], it[o]
    keep = np.ones(len(mz), bool)
    for j in range(1, len(mz)):
        if mz[j] - mz[j - 1] < 0.005:
            if it[j] >= it[j - 1]: keep[j - 1] = False
            else: keep[j] = False
    return mz[keep], it[keep]

@torch.no_grad()
def logits_batch(P, nets, dev, prec, adduct, instr, ce, mode):
    keep = [i for i, (a, _) in enumerate(P) if len(a)]
    if not keep: return None
    P = [P[i] for i in keep]
    B = len(P); N = max(len(a) for a, _ in P)
    mz = np.zeros((B, N), np.float32); it = np.zeros((B, N), np.float32); pad = np.ones((B, N), bool)
    for i, (a, b) in enumerate(P):
        mz[i, :len(a)] = a; it[i, :len(b)] = b; pad[i, :len(a)] = False
    T = lambda x: torch.as_tensor(x, device=dev)
    args = (T(mz), T(it), T(pad),
            T(np.asarray(prec, np.float32)[keep]),
            T(np.array([ADDUCT_IX.get(a, ADDUCT_IX['<unk>']) for a in np.asarray(adduct, object)[keep]])),
            T(np.array([instr_family(s) for s in np.asarray(instr, object)[keep]])),
            T(np.asarray(ce, np.float32)[keep]),
            T(np.asarray(mode, np.float32)[keep]))
    return np.mean([n(*args).float().mean(0).cpu().numpy() for n in nets], axis=0)

def molecule_logits(models, specs, prec, adduct, instr, ce, mode):
    single, merged, dev = models
    out = []
    if single:
        P = [prep_peaks(a, b, p) for (a, b), p in zip(specs, prec)]
        za = logits_batch(P, single, dev, prec, adduct, instr, ce, mode)
        if za is not None: out.append(za)
    if merged:
        mz, it = merge_peaks(specs)
        pm = float(np.median(prec))
        P = [prep_peaks(mz, it, pm)]
        zb = logits_batch(P, merged, dev, [pm], [adduct[0]], [instr[0]], [25.0], [float(np.mean(mode))])
        if zb is not None: out.append(zb)
    return np.mean(out, axis=0) if out else None

In [ ]:
# ---------- 加载模型与候选池 ----------
dev = torch.device("cpu")
single, merged, nbits = load_fp_models(
    [f"{FPD}/fp_single_s2.pt", f"{FPD}/fp_merged_m1.pt"], dev)
print(f"models: single={len(single)}, merged={len(merged)}, nbits={nbits}", flush=True)

pools = {}
for name in ["coco", "pubchemlite"]:
    fp = np.load(f"{FPD}/{name}_fp.npy", mmap_mode="r")
    mass = np.load(f"{FPD}/{name}_mass.npy")
    meta = pickle.load(open(f"{FPD}/{name}_meta.pkl", "rb"))
    pools[name] = dict(fp=fp, mass=mass, keys=meta["keys"], smiles=meta["smiles"])
    print(f"{name}: {fp.shape[0]} cands", flush=True)

# v17: expanded pools (ChEBI/LipidMaps + NPAtlas + HMDB)
if EXPD:
    for name, prefix in [("chebi_lm", "expanded_new"),
                         ("npatlas", "npatlas_new"),
                         ("hmdb", "hmdb_new")]:
        try:
            fp = np.load(f"{EXPD}/{prefix}_fp.npy", mmap_mode="r")
            mass = np.load(f"{EXPD}/{prefix}_mass.npy")
            meta = pickle.load(open(f"{EXPD}/{prefix}_meta.pkl", "rb"))
            pools[name] = dict(fp=fp, mass=mass, keys=meta["keys"], smiles=meta["smiles"])
            print(f"{name}: {fp.shape[0]} cands", flush=True)
        except Exception as e:
            print(f"WARNING: failed to load {name}: {e}", flush=True)
    print(f"v17 total pools: {list(pools.keys())}", flush=True)

In [ ]:
# ---------- 测试集与中性质量 ----------
ADDUCT_MAP = {
    "[M+H]+": (1, 1.007276, 1), "[M-H]-": (1, -1.007276, 1),
    "[M+Na]+": (1, 22.989219, 1), "[M+NH4]+": (1, 18.033823, 1),
    "[M+K]+": (1, 38.963158, 1), "[M]+": (1, 0.0, 1), "[M]-": (1, 0.0, 1),
    "[2M+H]+": (2, 1.007276, 1), "[2M+Na]+": (2, 22.989219, 1),
    "[2M-H]-": (2, -1.007276, 1), "[2M+NH4]+": (2, 18.033823, 1),
    "[2M+K]+": (2, 38.963158, 1), "[M+Cl]-": (1, 34.968853, 1),
    "[M+CH2O2-H]-": (1, 44.998204, 1), "[M+C2H4O2-H]-": (1, 59.013854, 1),
    "[M-H2O+H]+": (1, -17.003289, 1), "[M-2H2O+H]+": (1, -35.013854, 1),
    "[M+2H]2+": (1, 2.014552, 2), "[M+3H]3+": (1, 3.021828, 3),
    "[M-2H]-": (1, -2.014552, 1), "[M-H2O-H]-": (1, -18.010565, 1),
    "[2M+CH2O2-H]-": (2, 44.998204, 1), "[2M+C2H4O2-H]-": (2, 59.013854, 1),
    "[2M+Na-2H]-": (2, 20.974668, 1), "[M+Na-2H]-": (1, 20.974668, 1),
    "[M-H2O]+": (1, -18.010565, 1),
}
def neutral_mass(pmz, ad):
    m = ADDUCT_MAP.get(ad)
    if m is None or pmz is None or (isinstance(pmz, float) and np.isnan(pmz)): return np.nan
    n, d, z = m
    return (pmz * z - d) / n

te = pq.read_table(f"{DATA}/test.parquet").to_pandas()
te["neutral"] = [neutral_mass(mz, ad) for mz, ad in zip(te["precursor_mz"].values, te["adduct"].values)]
print(f"test: {len(te)} spectra, {te['molecule_id'].nunique()} molecules", flush=True)

# 训练库（保底填充用）
tr = pq.read_table(f"{DATA}/train.parquet",
    columns=["inchikey14", "normalized_smiles", "precursor_mz", "adduct"]).to_pandas()
tr["neutral"] = [neutral_mass(mz, ad) for mz, ad in zip(tr["precursor_mz"].values, tr["adduct"].values)]
tr = tr[tr["neutral"].notna()]
key2smiles = tr.groupby("inchikey14")["normalized_smiles"].agg(lambda s: s.value_counts().index[0])
lib = tr.groupby("inchikey14")["neutral"].median()
popular_keys = tr["inchikey14"].value_counts().index.tolist()
print(f"library: {len(lib)} structures", flush=True)

In [ ]:
# ---------- 主循环：指纹预测 + fz 排序 ----------
PPM = 10.0
models = (single, merged, dev)
mol_rows = []
mids = te["molecule_id"].unique()
for n, mid in enumerate(mids):
    g = te[te["molecule_id"] == mid]
    specs = list(zip(g["ms2_mzs"].values, g["ms2_normalized_intensities"].values))
    prec = g["precursor_mz"].values.astype(float)
    adduct = g["adduct"].values
    instr = g["instrument_type"].values if "instrument_type" in g else [None] * len(g)
    ce = np.full(len(g), 25.0, np.float32)
    if "collision_energy" in g:
        try: ce = g["collision_energy"].values.astype(float)
        except Exception: pass
    ce = np.where(np.isfinite(ce), ce, 25.0)
    mode = np.where(g["ionization_mode"].values == "positive", 1.0, -1.0).astype(np.float32)

    z = molecule_logits(models, specs, prec, adduct, instr, ce, mode)
    nm = np.nanmedian([neutral_mass(p, a) for p, a in zip(prec, adduct)])

    out, seen = [], set()
    def add(smiles, key):
        if key not in seen:
            seen.add(key); out.append(smiles)

    if z is not None and np.isfinite(nm):
        lo, hi = nm * (1 - PPM / 1e6), nm * (1 + PPM / 1e6)
        scored = []  # (fz, key, smiles)
        for name in pools:  # v17: all pools (coco, pubchemlite, chebi_lm, npatlas, hmdb)
            P = pools[name]
            a = np.searchsorted(P["mass"], lo, "left")
            b = np.searchsorted(P["mass"], hi, "right")
            if b <= a: continue
            bits = np.unpackbits(np.asarray(P["fp"][a:b]), axis=1)[:, :nbits].astype(np.float32)
            fz = bits @ z
            kk = P["keys"][a:b]; ss = P["smiles"][a:b]
            for score, k, s in zip(fz, kk, ss):
                scored.append((float(score), str(k), str(s)))
        scored.sort(key=lambda x: -x[0])
        for score, k, s in scored:
            if len(out) >= 25: break
            add(s, k)

    # 保底：训练库质量匹配 + 热门结构
    if len(out) < 25 and np.isfinite(nm):
        cand = lib[(lib >= nm * (1 - 35e-6)) & (lib <= nm * (1 + 35e-6))].index.tolist()
        for k in cand:
            if len(out) >= 25: break
            add(key2smiles[k], k)
    for k in popular_keys:
        if len(out) >= 25: break
        if k not in seen: add(key2smiles[k], k)

    mol_rows.append((mid, ";".join(out[:25])))
    if (n + 1) % 50 == 0: print(f"{n+1}/400 in {time.time()-t1:.1f}s", flush=True)

sub = pd.DataFrame(mol_rows, columns=["molecule_id", "smiles"])
sub.to_csv("/kaggle/working/submission.csv", index=False)
assert len(sub) == 400 and (sub["smiles"].str.split(";").str.len() == 25).all()
print(f"done in {time.time()-t1:.1f}s", flush=True)